In [1]:
import os
import csv
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torchvision import models
import torchvision.transforms as transforms
import numpy as np
from PIL import Image
from tqdm import tqdm

import random

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [2]:
#-------------读取训练集,训练集地址已经设定好，下面这段不用修改------------------#
#-----Read the training set, the address of the training set has been set, and the following section does not need to be modified-------#
train_path = "/bohr/train-i5ob/v1"
# train_path = "./"

In [3]:
all_set = set()
for a in range(10):
    for b in range(10):
        for c in range(10):
            for d in range(10):
                all_set.add((a + b + c + d, a * b * c * d))

all_map = {x : id for id, x in enumerate(all_set)}
all_id = {all_map[key] : key for key in all_map}

In [13]:
# 读取数据。
def load_train_data(data_dir='./train/'):
    label_path = os.path.join(data_dir, 'train_labels.csv')
    image_dir = os.path.join(data_dir, 'train_images')

    data = []
    with open(label_path, 'r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            sample_id = row['id']
            data.append({
                'id': sample_id,
                'image_path': os.path.join(image_dir, f'{sample_id}.png'),
                'sum': float(row['sum']),
                'product': float(row['product']),
            })

    print(f'Successfully loaded training records: {len(data)}')
    return data

def custom_transform(image):
    if len(image.shape) != 2:
        raise ValueError("Input image must be a grayscale image.")

    # 裁剪为 4 个 28x28 的图像
    crops = [
        image[:, 0:28],
        image[:, 28:56],
        image[:, 56:84],
        image[:, 84:112]
    ]

    # 定义变换
    transform = transforms.Compose([
        transforms.RandomRotation(degrees=15),  # 随机旋转
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),  # 随机平移和缩放
        transforms.ColorJitter(brightness=0.2, contrast=0.2),  # 随机调整亮度和对比度
    ])

    # 应用变换并存储结果
    transformed_crops = []
    for crop in crops:
        # 将裁剪的图像转换为 PIL 格式
        crop_pil = Image.fromarray(crop)
        # 应用变换
        transformed_crop = transform(crop_pil)
        transformed_crops.append(transformed_crop)

    # 随机打乱顺序
    random.shuffle(transformed_crops)

    # 拼接回 112x28 的图像
    stitched_image = np.concatenate([np.array(crop) for crop in transformed_crops], axis=1)

    # 转换为张量并归一化
    tensor_image = transforms.ToTensor()(stitched_image)
    normalized_image = transforms.Normalize(mean=(0.1307,), std=(0.3081,))(tensor_image)

    return normalized_image

def load_image(image_path):
    # 使用 PIL 读取灰度图像
    image = Image.open(image_path).convert('L')  # 转换为灰度图像
    image = np.array(image)  # 转换为 NumPy 数组

    # 确保图像大小为 112x28
    if image.shape != (28, 112):
        raise ValueError("Input image must be of size 112x28.")

    # 应用自定义变换
    transformed_image = custom_transform(image)
    
    return transformed_image

class MNISTBaselineDataset(Dataset):

    def __init__(self, records, train=True):
        self.records = records
        self.train=train

    def __len__(self):
        return len(self.records)

    @staticmethod
    def load_image(image_path):
        image = Image.open(image_path).convert('L')
        image = np.array(image, dtype=np.float32) / 255.0
        image = (image - 0.1307) / 0.3081
        return image

    def __getitem__(self, idx):
        record = self.records[idx]
        if self.train:
            image = load_image(record['image_path'])
        else:
            image = self.load_image(record['image_path'])
            image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        target = torch.tensor(all_map[(record['sum'], record['product'])], dtype=torch.long)
        return image, target


def create_train_loader(batch_size=64, num_workers=1, data_dir='./train/'):
    records = load_train_data(data_dir)
    
    # 使用 train_test_split 划分数据集
    train_records, valid_records = train_test_split(records, test_size=0.2, random_state=42)
    
    # 创建训练和验证数据集
    train_dataset = MNISTBaselineDataset(train_records)
    all_dataset = MNISTBaselineDataset(records)
    valid_dataset = MNISTBaselineDataset(valid_records, False)
    
    # 创建 DataLoader
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
    )
    all_loader = DataLoader(
        all_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
    )
    
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )
    
    return train_loader, all_loader, valid_loader

In [5]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.backbone = models.efficientnet_b0(weights=None)
        self.backbone.features[0][0] = nn.Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        self.backbone.classifier[1] = nn.Linear(self.backbone.classifier[1].in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)

In [15]:
# 训练模型
def train(model, train_loader, epochs, device='cpu'):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-5)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for images, targets in tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}', leave=False):
            images = images.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            preds = model(images)
            loss = criterion(preds, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * images.size(0)

        avg_loss = total_loss / len(train_loader.dataset)
        print(f'Epoch {epoch}/{epochs} - Loss: {avg_loss:.4f}')

def validate(model, valid_loader, device='cpu'):
    model.eval()  # 设置模型为评估模式
    correct = 0
    total = 0
    c0, c1 = 0, 0

    with torch.no_grad():  # 在验证时不需要计算梯度
        for images, targets in tqdm(valid_loader, desc='Validating', leave=False):
            images = images.to(device)
            targets = targets.to(device)

            preds = model(images)  # 获取模型预测
            _, predicted = torch.max(preds, 1)  # 获取预测的类别
            total += targets.size(0)  # 更新总样本数
            correct += (predicted == targets).sum().item()  # 更新正确预测的数量

            for p, t in zip(predicted.cpu().numpy(), targets.cpu().numpy()):
                p = int(p)
                t = int(t)
                if all_id[p][0] == all_id[t][0]:
                    c0 += 1
                if all_id[p][1] == all_id[t][1]:
                    c1 += 1

    accuracy = correct / total
    acc0 = c0 / total
    acc1 = c1 / total
    print(f'Validation Accuracy: {accuracy:.4f} | Validation Accuracy 0: {acc0:.4f} | Validation Accuracy 1: {acc1:.4f}')

def train_validate(model, train_loader, valid_loader, epochs, device='cpu'):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-5)

    for epoch in range(1, epochs + 1):
        # Training phase
        model.train()
        total_loss = 0.0

        for images, targets in tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}', leave=False):
            images = images.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            preds = model(images)
            loss = criterion(preds, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * images.size(0)

        avg_loss = total_loss / len(train_loader.dataset)
        print(f'Epoch {epoch}/{epochs} - Loss: {avg_loss:.4f}')

        # Validation phase
        model.eval()
        correct = 0
        total = 0
        c0, c1 = 0, 0

        with torch.no_grad():
            for images, targets in tqdm(valid_loader, desc='Validating', leave=False):
                images = images.to(device)
                targets = targets.to(device)

                preds = model(images)
                _, predicted = torch.max(preds, 1)

                total += targets.size(0)
                correct += (predicted == targets).sum().item()

                for p, t in zip(predicted.cpu().numpy(), targets.cpu().numpy()):
                    p = int(p)
                    t = int(t)
                    if all_id[p][0] == all_id[t][0]:
                        c0 += 1
                    if all_id[p][1] == all_id[t][1]:
                        c1 += 1

        accuracy = correct / total
        acc0 = c0 / total
        acc1 = c1 / total
        print(f'Validation Accuracy: {accuracy:.4f} | Validation Accuracy 0: {acc0:.4f} | Validation Accuracy 1: {acc1:.4f}')


def set_random_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

In [9]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed = 42
batch_size = 64
epochs = 30
set_random_seed(seed)

In [10]:
train_loader, all_loader, valid_loader = create_train_loader(batch_size=batch_size, data_dir=train_path)
model = SimpleCNN(len(all_set))
train_validate(model, all_loader, valid_loader, epochs=epochs, device=str(device))

In [16]:
# train_loader, valid_loader = create_train_loader(batch_size=batch_size, data_dir=train_path)
# validate(model, valid_loader, device=str(device))

In [18]:
#-------------读取测试集---------------#“DATA_PATH”是测试集加密后的环境变量，按照如下方式可以在提交后，系统评分时访问测试集，但是选手无法直接下载
#----Read the testing set, “DATA_PATH” is an environment variable for the encrypted test set. After submission, you can access the test set for system scoring in the following manner, but the contestant cannot download it directly.-----#
if os.environ.get('DATA_PATH'):
    test_path = os.environ.get("DATA_PATH") + "/"
else:
    test_path = "/bohr/train-i5ob/v1"
    print("Baseline 运行时，因为无法读取测试集，所以会有此条报错，属于正常现象")
    print("When baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.")
    #Baseline 运行时，因为无法读取测试集，所以会有此条报错，属于正常现象
    #When baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.

In [19]:
# 读取测试数据
class MNISTTestDataset(Dataset):
    def __init__(self, image_dir):
        self.image_paths = sorted(str(path) for path in Path(image_dir).glob('*.png'))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        sample_id = Path(image_path).stem
        image = MNISTBaselineDataset.load_image(image_path)
        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        return image, sample_id


# 这里用训练好的模型直接回归 sum 和 product。
def predict_and_save(model, image_dir, output_file, device='cpu', batch_size=64, num_workers=1):
    dataset = MNISTTestDataset(image_dir)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    rows = []
    model.eval()
    with torch.no_grad():
        for images, sample_ids in tqdm(loader, desc=f'Predicting {os.path.basename(image_dir)}', leave=False):
            images = images.to(device)
            preds = model(images).argmax(dim=-1).cpu().numpy()

            for sample_id, pred_id in zip(sample_ids, preds):
                rows.append({
                    'id': sample_id,
                    'sum': all_id[pred_id][0],
                    'product': all_id[pred_id][1],
                })

    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['id', 'sum', 'product'])
        writer.writeheader()
        writer.writerows(rows)

    print(f'Saved predictions to {output_file}')


val_dir = os.path.join(test_path, 'val')
test_dir = os.path.join(test_path, 'test')
# val_dir = os.path.join(test_path, 'train_images')3
# test_dir = os.path.join(test_path, 'train_images')

for required_dir in [val_dir, test_dir]:
    if not os.path.isdir(required_dir):
        raise FileNotFoundError(f'Missing test directory: {required_dir}')

predict_and_save(model, val_dir, 'submission_val.csv', device=str(device), batch_size=batch_size)
predict_and_save(model, test_dir, 'submission_test.csv', device=str(device), batch_size=batch_size)

In [ ]:
import zipfile

# 定义要打包的文件和压缩文件名
files_to_zip = ['submission_val.csv', 'submission_test.csv']
zip_filename = 'submission.zip'

# 创建一个 zip 文件
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        # 将文件添加到 zip 文件中
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} 创建成功!')